# Teste do `run_inference_local_HF.py` (transformers + log-probabilidades)

Notebook de teste, autónomo. Antes: **Runtime → GPU (T4)**. Corre as células por ordem (ou Run All).
O modelo de teste (`Qwen/Qwen2.5-3B-Instruct`) não precisa de token nem de aprovação. A primeira vez descarrega o modelo, o que demora uns minutos.

In [ ]:
!nvidia-smi

In [ ]:
# Clonar a branch (se ainda não estiver clonada nesta máquina do Colab)
REPO_URL = "https://github.com/Luis-FO/Prompt-Engineering-for-KE.git"
BRANCH = "rui/colab"
MODEL = "Qwen/Qwen2.5-3B-Instruct"

import os
os.chdir("/content")
if not os.path.isdir("kl"):
    !git clone -b {BRANCH} {REPO_URL} kl
os.chdir("/content/kl")
!ls 2_inference_module

In [ ]:
# Libertar a GPU, se o Ollama ainda estiver a correr nesta máquina (dá erro inofensivo se não estiver)
!pkill ollama || true

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# Ficheiro de teste com os 3 primeiros prompts do three_shot_RAG
import json
SRC = "1_data_module/2_creating_prompts/data_with_prompts/three_shot_RAG_prompts_test.json"
os.makedirs("/content/prompts_teste", exist_ok=True)
json.dump(json.load(open(SRC))[:3], open("/content/prompts_teste/three_shot_RAG_prompts_test.json", "w"))

In [ ]:
!python 2_inference_module/run_inference_local_HF.py --model_name {MODEL} --input_folder /content/prompts_teste --out_dir /content/out_hf_teste --batch_size 3

In [ ]:
# Ver a resposta e, token a token, a log-probabilidade (perto de 0 = confiante; muito negativo = inseguro)
import json
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
d = json.load(open("/content/out_hf_teste/three_shot_RAG_prompts_test.json"))
print("respostas:", len(d))
x = d[0]
print(x["generated_text"])
print()
for t, lp in list(zip(x["token_ids"], x["token_logprobs"]))[:40]:
    print(repr(tok.decode([t])), round(lp, 3))

In [ ]:
# Verificações rápidas
for x in d:
    assert len(x["token_ids"]) == len(x["token_logprobs"])
    assert all(lp <= 1e-6 for lp in x["token_logprobs"])
    media = sum(x["token_logprobs"]) / max(1, len(x["token_logprobs"]))
    print(f"prompt {x['index']}: {len(x['token_ids'])} tokens, logprob média {media:.3f}")
print("ok")

In [ ]:
# Corrida completa (446 prompts) + post-processamento + avaliação. Resultados no Google Drive.
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/kl_out_" + MODEL.split("/")[-1]
os.chdir("/content/kl")

!python 2_inference_module/run_inference_local_HF.py --model_name {MODEL} --out_dir {OUT} --batch_size 4
!python 2_inference_module/post_process_generated_string.py --input_folder {OUT} --out_dir {OUT}_pp
!python 3_evaluation_module/evaluation_reference_to_annotations/evaluate.py --input_dir {OUT}_pp --gold_file 1_data_module/1_data_preprocessing/RED-fm/test.jsonl --out_dir {OUT}_eval